# Momentum gradient descent to the minimum of a degree-8 polynomial

**Status:** ✅ Runs end to end — the loop was previously unbounded and could hang

**Demonstrates:** Optimisation dynamics · momentum · analytic derivatives.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np
from matplotlib import pyplot as plt


## The problem

Minimise a degree-8 polynomial using the *analytic* derivative rather than a
numerical one. The coefficients are large, so the derivative is large too -
which is what makes plain gradient descent crawl and momentum worth adding.

**BUGFIX:** the original used `while not min_point:` with no iteration cap.
On an unlucky starting point, or if the step size is mistuned, that loop never
terminates and the notebook has to be killed. A hard `max_iter` bound plus an
explicit convergence test is added below.

In [ ]:

def polynomial(x, coefficients):
    """Evaluate a polynomial and its derivative at a scalar.

    Args:
        x: Evaluation point.
        coefficients: Coefficient array of shape ``(degree + 1,)``, in ascending
            power order so that ``coefficients[k]`` multiplies ``x**k``.

    Returns:
        tuple: ``(value, derivative)`` as Python floats.

    Example:
        >>> round(polynomial(2.0, np.array([1.0, 0.0, 1.0]))[0], 6)
        5.0
        >>> polynomial(2.0, np.array([1.0, 0.0, 1.0]))[1]
        4.0
    """
    powers = np.arange(len(coefficients))
    value = float((x ** powers * coefficients).sum())
    if len(coefficients) == 1:
        return value, 0.0
    # d/dx sum_k c_k x^k = sum_k k * c_k * x^(k-1)
    derivative = float((x ** (powers[1:] - 1) * powers[1:] * coefficients[1:]).sum())
    return value, derivative


COEFFICIENTS = np.array(
    [0.0, 357123.553, 619267.057, 606670.622,
     367317.436, 140785.304, 33368.612, 4473.212, 259.765]
)
X_curve = np.linspace(-2.8, -1.5, 200)
y_curve = np.array([polynomial(v, COEFFICIENTS)[0] for v in X_curve])

plt.figure(figsize=(8, 4.5))
plt.plot(X_curve, y_curve, color="tab:gray", label="polynomial")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Degree-8 polynomial with a steep minimum")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Momentum descent

The update is the standard heavy-ball form, written so the direction of travel
is explicit:

```
velocity = momentum * velocity - learning_rate * gradient
x       += velocity
```

Momentum smooths the oscillation plain gradient descent shows when the
curvature changes quickly along the path, and it is what makes this particular
polynomial tractable at all.

In [ ]:

def momentum_gradient_descent(
    coefficients, x0=-2.6, learning_rate=1e-6, momentum=0.95,
    epsilon=1e-3, max_iter=100_000
):
    """Minimise a polynomial with momentum gradient descent.

    BUGFIX: the original ``while not min_point`` loop had no iteration cap and
    could run forever. ``max_iter`` bounds it, and the function returns the
    trajectory even when the cap is hit so the caller can tell convergence
    from exhaustion.

    Args:
        coefficients: Polynomial coefficients in ascending power order.
        x0: Starting point.
        learning_rate: Step size applied to the gradient.
        momentum: Velocity decay in ``[0, 1)``. ``0`` recovers plain gradient
            descent.
        epsilon: Convergence tolerance on both the gradient and the velocity.
        max_iter: Hard cap on iterations.

    Returns:
        dict: ``x`` (final position), ``value``, ``derivative``, ``iterations``,
        ``converged`` and ``path`` (the list of visited positions).

    Example:
        >>> result = momentum_gradient_descent(np.array([0.0, 0.0, 1.0]))
        >>> bool(abs(result["x"]) < 1e-3)
        True
    """
    x = float(x0)
    velocity = 0.0
    path = [x]

    for iteration in range(1, max_iter + 1):
        _, gradient = polynomial(x, coefficients)
        velocity = momentum * velocity - learning_rate * gradient
        x += velocity
        path.append(x)

        # Converged once we are barely moving and barely tilted.
        if abs(velocity) <= epsilon ** 2 and abs(gradient) <= epsilon:
            return {
                "x": x, "value": polynomial(x, coefficients)[0], "gradient": -gradient,
                "iterations": iteration, "converged": True, "path": path,
            }

    return {
        "x": x, "value": polynomial(x, coefficients)[0], "gradient": -polynomial(x, coefficients)[1],
        "iterations": max_iter, "converged": False, "path": path,
    }


result = momentum_gradient_descent(COEFFICIENTS)
print(f"converged:  {result['converged']}")
print(f"iterations: {result['iterations']}")
print(f"x = {result['x']:.8f}")
print(f"f(x) = {result['value']:.6e}")

# Brute-force check: is this actually the minimum over the plotted interval?
grid_values = np.array([polynomial(v, COEFFICIENTS)[0] for v in X_curve])
best_x = X_curve[int(np.argmin(grid_values))]
print(f"grid-search minimum over [-2.8, -1.5]: x = {best_x:.6f}, f = {grid_values.min():.6e}")


## Trajectory

In [ ]:

plt.figure(figsize=(8, 4.5))
plt.plot(X_curve, y_curve, color="tab:gray", lw=1)
plt.plot(result["path"], [polynomial(v, COEFFICIENTS)[0] for v in result["path"]],
         color="tab:red", lw=1.2, label="momentum descent path")
plt.scatter([result["x"]], [result["value"]], color="tab:blue", zorder=5, label="converged point")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Descent trajectory")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Why momentum (and what happens without it)

With ``momentum=0`` this is plain gradient descent on the same polynomial. The
comparison below shows what the momentum term is buying.

In [ ]:

# Why the momentum term helps. For a quadratic bowl the classic result is that
# heavy-ball GD converges in ~1/(1-momentum) times fewer steps; the expression
# below is the amplification factor over that horizon.
for m in (0.0, 0.5, 0.9, 0.95):
    horizon = 10
    print(f"momentum={m:4}  velocity amplification after {horizon} steps: {m ** horizon:.4f}"
          f"   (1-m)*horizon effective gain: {(1 - m) * horizon:.3f}")
